Import libraries and functions

In [1]:
# Import

# Import Python standard libraries
import json  # Import json module for parsing JSON data
import logging  # Import logging module for implementing event logging
from pathlib import Path  # From pathlib module import Path class for handling file paths
from typing import Union, Dict, Any, List, Optional  # From typing module import relevant classes for type hinting in function signatures
from urllib.parse import urlparse  # From urllib.parse module import urlparse function for identifying URLs

# Import external packages
import requests  # Import Requests library for downloading through URLs
import networkx as nx  # Import NetworkX package for network creation

Configure logging

In [2]:
# Configure logging

logging.basicConfig(  # Initiate configuration
    level=logging.DEBUG,  # Set minimum severity level
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'  # Set logging to display timestamp, logger name, severity level, and message
)

logger = logging.getLogger('affiliation_builder')  # Create package logger instance

Define build() function

In [ ]:
# Define build() function

def build(  # Declare function
    # Define parameters    
    json_path: Union[str, Path],  # Take either string or Path object
    node_set_0_key: Optional[str],
    node_set_1_keys: Union[str, List[str]],  # Take either string or list of strings
    identifier_key: str,
    node_set_1_identifier_key: Optional[str] = None
) -> nx.Graph:  # Define return type
    """
    Build a bipartite affiliation network from JSON data.
    
    Creates a bipartite NetworkX graph where node set 0 (e.g., events) connects
    to node set 1 (e.g., human and organization participants in event) through
    affiliation relationships.
    All JSON fields are preserved as node attributes.
    
    Parameters
    ----------
    json_path : str or Path
        Path to JSON file or URL (http/https).
    node_set_0_key : str or None
        JSON key containing node-set-0 items (NetworkX bipartite=0). 
        Use None if JSON is a direct array without a wrapping key.
    node_set_1_keys : str or list of str
        JSON key(s) for affiliated entities (NetworkX bipartite=1) in each node-set-0
        item.
    identifier_key : str
        JSON key to use as unique identifier for node-set-0 items.
    node_set_1_identifier_key : str or None, default=None
        JSON key to use as identifier for node set 1 items when they are objects
        (rather than simple values).
        If None, assumes node set 1 items are simple values (strings/numbers).
        If provided, extracts this key from each entity object and preserves
        all other keys as node attributes.
                
    Returns
    -------
    networkx.Graph
        Bipartite graph with two node sets connected by affiliation edges.
        
    Examples
    --------
    >>> # Simple entities (strings)
    >>> G = build('events.json', 'events', 'participants', 'name')
    
    >>> # Complex entities (objects with metadata)
    >>> G = build(json_path='events.json',
    ...           node_set_0_key='events',
    ...           node_set_1_keys='participants',
    ...           identifier_key='name',
    ...           node_set_1_identifier_key='person_name')
    
    >>> # Direct array format
    >>> G = build('events.json', None, 'members', 'id')
    
    >>> # Multiple entity types with complex objects
    >>> G = build('events.json', 'events', ['persons', 'organizations'], 'name',
    ...           node_set_1_identifier_key='name')
    
    Notes
    -----
    Node set 0 has bipartite=0, node set 1 has bipartite=1. Edges only exist
    between nodes of different sets. Access node sets with:
    
        node_set_0 = {n for n, d in G.nodes(data=True) if d['bipartite'] == 0}
        node_set_1 = {n for n, d in G.nodes(data=True) if d['bipartite'] == 1}

    When node_set_1_identifier_key is specified, all other attributes from entity 
    objects are preserved as node attributes, enabling rich metadata analysis.

    """

    # Log function call and arguments
    logger.info(f"Affiliation network builder started")
    logger.info(f"JSON source: {json_path}")
    logger.info(f"Node-set-0 key: '{node_set_0_key}'")
    logger.info(f"Node-set-1 keys: {node_set_1_keys}")
    logger.info(f"Identifier key: '{identifier_key}'")
    logger.info(f"Node-set-1 identifier key: '{node_set_1_identifier_key}'")

    # =========================================================================
    # INPUT VALIDATION
    # =========================================================================
    # Validate input types
    
    # Validate node_set_1_keys type
    if not isinstance(node_set_1_keys, (str, list)):  # Check data type
        logger.error(
            f"Invalid type for node_set_1_keys detected: {type(node_set_1_keys)}"  # Log type error
        )
        raise TypeError(
            f"node_set_1_keys must be string or list, "
            f"but got {type(node_set_1_keys).__name__}"  # Raise type error
        )
    
    # Validate identifier_key type
    if not isinstance(identifier_key, str):
        logger.error(
            f"Invalid type for identifier_key detected: {type(identifier_key)}"
        )
        raise TypeError(
            f"identifier_key must be string, "
            f"but got {type(identifier_key).__name__}"
        )
    
    # Validate node_set_1_identifier_key type (if provided)
    if node_set_1_identifier_key is not None and not isinstance(node_set_1_identifier_key, str):
        logger.error(
            f"Invalid type for node_set_1_identifier_key detected: {type(node_set_1_identifier_key)}"
        )
        raise TypeError(
            f"node_set_1_identifier_key must be string or None, "
            f"but got {type(node_set_1_identifier_key).__name__}"
        )
    
    # Validate node_set_0_key type
    if node_set_0_key is not None and not isinstance(node_set_0_key, str):
        logger.error(
            f"Invalid type for node_set_0_key detected: {type(node_set_0_key)}"
        )
        raise TypeError(
            f"node_set_0_key must be string or None, "
            f"but got {type(node_set_0_key).__name__}"
        )
    
    logger.debug("Input validation passed")

    # =========================================================================
    # STEP 1: NORMALIZE node_set_1_keys TO A LIST
    # =========================================================================
    # Users can pass either a string or a list. Strings will be converted to
    # lists for later processing.
    # Example: 'members' becomes ['members']
    # Example: ['persons', 'organizations'] stays ['persons', 'organizations']
    
    if isinstance(node_set_1_keys, str):  # Detect string type
        node_set_1_keys = [node_set_1_keys]  # Convert to list
        logger.debug(f"Converting node_set_1_keys to list: {node_set_1_keys}")  # Log normalization

    # =========================================================================
    # STEP 2: LOAD JSON DATA (LOCAL FILE OR URL)
    # =========================================================================
    # Determine data source and load data.

    is_url = False  # Set local source as default

    # Check URL source
    if isinstance(json_path, str):  # Check string type
        parsed = urlparse(json_path)  # Parse URL
        is_url = parsed.scheme in ('http', 'https')  # Check URL scheme
    
    # Load from URL

    if is_url:
        logger.info(f"URL detected, downloading JSON from: {json_path}")  # Log URL detection
        try:
            response = requests.get(json_path, timeout=30)  # Send server request; set timeout to 30 seconds; return Response object
            response.raise_for_status()  # Raise exception for HTTP status code 4xx or 5xx
            data = response.json()  # Decode Response object as UTF-8 text; parse JSON into Python dictionary
            logger.info(f"JSON successfully downloaded") # Log URL download
                        
        except requests.exceptions.Timeout:  # Catch timeout error
            logger.error(f"Request timed out")  # Log timeout error
            raise requests.exceptions.Timeout(
                f"Request timed out while accessing {json_path}"  # Re-raise timeout error with added context
            )
        
        except requests.exceptions.ConnectionError as e:
            logger.error(f"Connection error: {e}")  # Log ConnectionError specifics
            raise requests.exceptions.ConnectionError(
                f"Could not connect to {json_path}"
            )
        
        except requests.exceptions.HTTPError as e:  # Catch HTTP error raised by response.raise_for_status()
            logger.error(f"HTTP error: {e}")
            raise  # Re-raise without added context (URL already part of original error message)

        

Test build

In [4]:
G = build(
    '../example.json',
    'events',
    ['persons', 'organizations'],
    'name',
    node_set_1_identifier_key='name'
)

2025-11-11 15:36:08,018 - affiliation_builder - INFO - Affiliation network builder started
2025-11-11 15:36:08,018 - affiliation_builder - INFO - JSON source: ../example.json
2025-11-11 15:36:08,019 - affiliation_builder - INFO - Node-set-0 key: 'events'
2025-11-11 15:36:08,019 - affiliation_builder - INFO - Node-set-1 keys: ['persons', 'organizations']
2025-11-11 15:36:08,019 - affiliation_builder - INFO - Identifier key: 'name'
2025-11-11 15:36:08,020 - affiliation_builder - INFO - Node-set-1 identifier key: 'name'
2025-11-11 15:36:08,020 - affiliation_builder - DEBUG - Input validation passed
